In [1]:
import os
import numpy as np
import scipy.io as sio

# Dataset path
DATASET_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\dataset\raw"

# Locate Part 1
part1_path = os.path.join(DATASET_PATH, "part_1")

# Get MATLAB files
mat_files = sorted([
    f for f in os.listdir(part1_path)
    if f.lower().endswith(".mat")
])

print("MATLAB files in Part 1:", len(mat_files))

# Load first MRI record
sample_path = os.path.join(part1_path, mat_files[0])
data = sio.loadmat(sample_path, simplify_cells=True)

cjdata = data["cjdata"]

print("\nFILE:", mat_files[0])
print("AVAILABLE FIELDS:", list(cjdata.keys()))

# Extract image and mask
image = np.asarray(cjdata["image"])
mask = np.asarray(cjdata["tumorMask"])

print("\nIMAGE INFORMATION")
print("Shape:", image.shape)
print("Data type:", image.dtype)
print("Minimum:", image.min())
print("Maximum:", image.max())

print("\nTUMOR INFORMATION")
print("Label:", cjdata["label"])
print("Patient ID:", cjdata["PID"])
print("Mask shape:", mask.shape)
print("Mask values:", np.unique(mask))

print("\nDATASET INSPECTION COMPLETED")

MATLAB files in Part 1: 766


NotImplementedError: Please use HDF reader for matlab v7.3 files, e.g. h5py

In [ ]:
import h5py
import os

DATASET_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\dataset\raw"

part1_path = os.path.join(DATASET_PATH, "part_1")

mat_files = sorted([
    f for f in os.listdir(part1_path)
    if f.endswith(".mat")
])

sample_path = os.path.join(part1_path, mat_files[0])

print("Sample file:", mat_files[0])

with h5py.File(sample_path, "r") as f:

    print("\nROOT KEYS:")
    print(list(f.keys()))

    print("\nDATASET STRUCTURE:")

    def inspect_structure(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(name, "| Shape:", obj.shape, "| Data type:", obj.dtype)
        else:
            print(name, "| Group")

    f.visititems(inspect_structure)

In [ ]:

import h5py
import numpy as np
import matplotlib.pyplot as plt
import os

# Load first MRI record
with h5py.File(sample_path, "r") as f:

    cjdata = f["cjdata"]

    image = np.array(cjdata["image"])
    mask = np.array(cjdata["tumorMask"])

    label = int(np.array(cjdata["label"])[0, 0])

    # Decode patient ID
    pid_raw = np.array(cjdata["PID"]).flatten()
    patient_id = "".join(chr(int(x)) for x in pid_raw)

    border = np.array(cjdata["tumorBorder"])

# Class mapping
class_names = {
    1: "Meningioma",
    2: "Glioma",
    3: "Pituitary Tumor"
}

print("PATIENT ID:", patient_id)
print("TUMOR LABEL:", label)
print("TUMOR CLASS:", class_names[label])

print("\nIMAGE SHAPE:", image.shape)
print("IMAGE MINIMUM:", image.min())
print("IMAGE MAXIMUM:", image.max())

print("\nMASK SHAPE:", mask.shape)
print("MASK UNIQUE VALUES:", np.unique(mask))
print("TUMOR PIXELS:", np.count_nonzero(mask))

# Visualize MRI and tumor mask
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("Original Brain MRI")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Tumor Mask")
axes[1].axis("off")

axes[2].imshow(image, cmap="gray")
axes[2].imshow(mask, cmap="autumn", alpha=0.4)
axes[2].set_title("MRI with Tumor Mask Overlay")
axes[2].axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import os
import h5py
import numpy as np
import pandas as pd
from collections import Counter

DATASET_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\dataset\raw"

class_names = {
    1: "Meningioma",
    2: "Glioma",
    3: "Pituitary Tumor"
}

records = []
errors = []

for part in range(1, 5):

    folder = os.path.join(DATASET_PATH, f"part_{part}")

    mat_files = sorted([
        f for f in os.listdir(folder)
        if f.lower().endswith(".mat")
    ])

    print(f"Inspecting Part {part}: {len(mat_files)} files")

    for filename in mat_files:

        file_path = os.path.join(folder, filename)

        try:
            with h5py.File(file_path, "r") as f:

                cjdata = f["cjdata"]

                image_shape = cjdata["image"].shape
                mask_shape = cjdata["tumorMask"].shape

                label = int(cjdata["label"][0, 0])

                pid_raw = np.array(cjdata["PID"]).flatten()
                patient_id = "".join(
                    chr(int(x)) for x in pid_raw
                )

                records.append({
                    "Part": part,
                    "Filename": filename,
                    "Patient_ID": patient_id,
                    "Label": label,
                    "Class": class_names.get(label, "Unknown"),
                    "Image_Shape": image_shape,
                    "Mask_Shape": mask_shape
                })

        except Exception as e:
            errors.append({
                "File": filename,
                "Part": part,
                "Error": str(e)
            })

df = pd.DataFrame(records)

print("\n" + "=" * 60)
print("FULL DATASET INSPECTION SUMMARY")
print("=" * 60)

print("Total readable records:", len(df))
print("Total errors:", len(errors))

print("\nCLASS DISTRIBUTION:")
print(df["Class"].value_counts())

print("\nUNIQUE PATIENTS:", df["Patient_ID"].nunique())

print("\nIMAGE DIMENSIONS:")
print(df["Image_Shape"].value_counts())

print("\nMASK DIMENSIONS:")
print(df["Mask_Shape"].value_counts())

print("\nDUPLICATE PATIENT RECORD SUMMARY:")
print(df.groupby("Patient_ID").size().describe())

print("\nINVALID LABELS:")
print(df[~df["Label"].isin([1, 2, 3])])

print("\nINSPECTION COMPLETED")


In [ ]:
# Identify records with dimensions different from 512 × 512

small_images = df[
    df["Image_Shape"] != (512, 512)
].copy()

print("Number of smaller images:", len(small_images))

print("\nSMALL IMAGE RECORDS")
display(
    small_images[
        ["Part", "Filename", "Patient_ID", "Class",
         "Image_Shape", "Mask_Shape"]
    ].sort_values(["Part", "Filename"])
)

print("\nCLASS DISTRIBUTION OF SMALL IMAGES")
print(small_images["Class"].value_counts())

print("\nUNIQUE PATIENTS WITH SMALL IMAGES:")
print(small_images["Patient_ID"].nunique())

In [ ]:
# Investigate all records belonging to patient 108931

patient_records = df[
    df["Patient_ID"] == "108931"
].copy()

print("PATIENT ID: 108931")
print("=" * 50)

print("Total MRI records:", len(patient_records))

print("\nIMAGE DIMENSIONS:")
print(patient_records["Image_Shape"].value_counts())

print("\nTUMOR CLASSES:")
print(patient_records["Class"].value_counts())

print("\nRECORD DETAILS:")

display(
    patient_records[
        ["Part", "Filename", "Image_Shape", "Mask_Shape", "Class"]
    ].sort_values(["Part", "Filename"])
)

In [ ]:
# Analyze tumor classes at patient level

patient_summary = (
    df.groupby("Patient_ID")
    .agg(
        Number_of_Images=("Filename", "count"),
        Number_of_Classes=("Label", "nunique"),
        Tumor_Class=("Class", "first")
    )
    .reset_index()
)

print("TOTAL UNIQUE PATIENTS:", len(patient_summary))

print("\nPATIENT DISTRIBUTION BY TUMOR CLASS:")
print(patient_summary["Tumor_Class"].value_counts())

print("\nIMAGE DISTRIBUTION BY PATIENT CLASS:")
print(
    patient_summary.groupby("Tumor_Class")["Number_of_Images"]
    .agg(["count", "sum", "mean", "min", "max"])
)

print("\nPATIENTS WITH MULTIPLE TUMOR LABELS:")
print(
    patient_summary[
        patient_summary["Number_of_Classes"] > 1
    ]
)

print("\nPATIENT IMAGE COUNT SUMMARY:")
print(patient_summary["Number_of_Images"].describe())

In [ ]:
import h5py
import scipy.io as sio
import os

cvind_path = os.path.join(DATASET_PATH, "cvind.mat")

print("CVIND FILE:", cvind_path)
print("File size:", round(os.path.getsize(cvind_path) / 1024, 2), "KB")

if h5py.is_hdf5(cvind_path):

    print("\nMATLAB v7.3 FORMAT")

    with h5py.File(cvind_path, "r") as f:

        print("ROOT KEYS:", list(f.keys()))

        def inspect_cv(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(name, "| Shape:", obj.shape, "| Type:", obj.dtype)
            else:
                print(name, "| Group")

        f.visititems(inspect_cv)

else:

    print("\nMATLAB CLASSIC FORMAT")

    print("VARIABLES:")
    print(sio.whosmat(cvind_path))

    cv_data = sio.loadmat(cvind_path)

    for key, value in cv_data.items():
        if not key.startswith("__"):
            print(key, "| Shape:", getattr(value, "shape", None),
                  "| Type:", type(value))

In [ ]:
import h5py
import numpy as np
from collections import Counter

cvind_path = os.path.join(DATASET_PATH, "cvind.mat")

with h5py.File(cvind_path, "r") as f:

    cvind = np.array(f["cvind"]).flatten()

print("CVIND SHAPE:", cvind.shape)
print("DATA TYPE:", cvind.dtype)

print("\nFIRST 30 VALUES:")
print(cvind[:30])

print("\nLAST 30 VALUES:")
print(cvind[-30:])

print("\nUNIQUE FOLD VALUES:")
print(np.unique(cvind))

print("\nFOLD DISTRIBUTION:")

fold_counts = Counter(cvind.astype(int))

for fold, count in sorted(fold_counts.items()):
    print(f"Fold {fold}: {count} images")

print("\nTOTAL ASSIGNMENTS:", len(cvind))
print("INVALID OR NON-INTEGER VALUES:",
      np.sum(cvind != np.floor(cvind)))

In [ ]:
# Match cvind assignments to records using numeric filenames.
# The original dataset files are numbered 1.mat through 3064.mat.

df_cv = df.copy()

df_cv["File_Number"] = df_cv["Filename"].str.replace(
    ".mat", "", regex=False
).astype(int)

df_cv = df_cv.sort_values("File_Number").reset_index(drop=True)

# Verify that file numbering matches cvind positions
assert len(df_cv) == len(cvind)
assert df_cv["File_Number"].tolist() == list(range(1, 3065))

df_cv["Fold"] = cvind.astype(int)

# Count how many different folds contain each patient
patient_fold_counts = (
    df_cv.groupby("Patient_ID")["Fold"]
    .nunique()
)

leaking_patients = patient_fold_counts[
    patient_fold_counts > 1
]

print("OFFICIAL FIVE-FOLD PATIENT LEAKAGE AUDIT")
print("=" * 55)

print("Total patients:", df_cv["Patient_ID"].nunique())
print("Patients appearing in multiple folds:", len(leaking_patients))
print("Patients confined to one fold:",
      (patient_fold_counts == 1).sum())

print("\nPATIENTS WITH FOLD LEAKAGE:")

if len(leaking_patients) > 0:
    print(leaking_patients.sort_values(ascending=False))
else:
    print("No patient leakage detected.")

print("\nIMAGE DISTRIBUTION BY FOLD:")
print(df_cv["Fold"].value_counts().sort_index())

In [ ]:
# Class distribution across the official five folds

print("IMAGE DISTRIBUTION BY CLASS AND FOLD")
print("=" * 60)

image_distribution = pd.crosstab(
    df_cv["Class"],
    df_cv["Fold"],
    margins=True
)

display(image_distribution)

print("\nPATIENT DISTRIBUTION BY CLASS AND FOLD")
print("=" * 60)

patient_distribution = pd.crosstab(
    patient_summary["Tumor_Class"],
    df_cv[["Patient_ID", "Fold"]]
    .drop_duplicates()
    .set_index("Patient_ID")["Fold"],
    margins=True
)

display(patient_distribution)

print("\nCLASS PROPORTIONS IN EACH FOLD")
print("=" * 60)

class_proportions = pd.crosstab(
    df_cv["Fold"],
    df_cv["Class"],
    normalize="index"
) * 100

display(class_proportions.round(2))

In [ ]:
# Count unique patients by class and official fold

patient_df = (
    df_cv[["Patient_ID", "Class", "Fold"]]
    .drop_duplicates(subset=["Patient_ID"])
)

patient_distribution = pd.crosstab(
    patient_df["Class"],
    patient_df["Fold"],
    margins=True
)

print("PATIENT DISTRIBUTION BY CLASS AND FOLD")
print("=" * 60)

display(patient_distribution)

print("\nTOTAL UNIQUE PATIENTS:", patient_df["Patient_ID"].nunique())

In [ ]:
# Save verified dataset metadata

import os

METADATA_PATH = os.path.join(
    r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI",
    "results",
    "verified_dataset_metadata.csv"
)

df_cv.to_csv(METADATA_PATH, index=False)

print("Metadata successfully saved.")
print("Location:", METADATA_PATH)
print("Records:", len(df_cv))
print("Unique patients:", df_cv["Patient_ID"].nunique())
print("Unique folds:", df_cv["Fold"].nunique())

In [ ]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt

# Keep Class as a normal column
sample_records = (
    df_cv.groupby("Class")
    .first()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

print("MRI IMAGE INTENSITY INSPECTION")
print("=" * 55)

for ax, (_, record) in zip(axes, sample_records.iterrows()):

    part_folder = f"part_{int(record['Part'])}"

    file_path = os.path.join(
        DATASET_PATH,
        part_folder,
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        image = np.array(f["cjdata"]["image"]).astype(np.float32)

    print(f"\nClass: {record['Class']}")
    print("Patient ID:", record["Patient_ID"])
    print("Image shape:", image.shape)
    print("Minimum:", image.min())
    print("Maximum:", image.max())
    print("Mean:", round(float(image.mean()), 2))
    print("Standard deviation:", round(float(image.std()), 2))
    print("Unique intensity values:", len(np.unique(image)))

    ax.imshow(image, cmap="gray")
    ax.set_title(record["Class"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import h5py
import os

# Reproducible sample of 100 images
sample_df = df_cv.sample(n=100, random_state=42)

intensity_results = []

for _, record in sample_df.iterrows():

    file_path = os.path.join(
        DATASET_PATH,
        f"part_{int(record['Part'])}",
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        image = np.array(f["cjdata"]["image"], dtype=np.float32)

    intensity_results.append({
        "Class": record["Class"],
        "Patient_ID": record["Patient_ID"],
        "Minimum": image.min(),
        "Maximum": image.max(),
        "Mean": image.mean(),
        "Std": image.std(),
        "P50": np.percentile(image, 50),
        "P95": np.percentile(image, 95),
        "P99": np.percentile(image, 99),
        "P99.5": np.percentile(image, 99.5)
    })

intensity_df = pd.DataFrame(intensity_results)

print("INTENSITY DISTRIBUTION SUMMARY")
print("=" * 65)

display(
    intensity_df.groupby("Class")[
        ["Minimum", "Maximum", "Mean", "Std",
         "P50", "P95", "P99", "P99.5"]
    ].median().round(2)
)

print("\nNUMBER OF SAMPLED IMAGES:", len(intensity_df))
print("UNIQUE PATIENTS:", intensity_df["Patient_ID"].nunique())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tumor_classes = [
    "Glioma",
    "Meningioma",
    "Pituitary Tumor"
]

for ax, tumor_class in zip(axes, tumor_classes):

    class_data = intensity_df[
        intensity_df["Class"] == tumor_class
    ]

    ax.hist(
        class_data["Maximum"],
        bins=15,
        alpha=0.7,
        edgecolor="black"
    )

    ax.set_title(f"{tumor_class} - Maximum Intensity")
    ax.set_xlabel("Maximum Pixel Intensity")
    ax.set_ylabel("Number of Images")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, tumor_class in zip(axes, tumor_classes):

    class_data = intensity_df[
        intensity_df["Class"] == tumor_class
    ]

    ax.hist(
        class_data["P99"],
        bins=15,
        alpha=0.7,
        edgecolor="black"
    )

    ax.set_title(f"{tumor_class} - 99th Percentile")
    ax.set_xlabel("P99 Intensity")
    ax.set_ylabel("Number of Images")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os

# Select one image from each tumor class
sample_records = (
    df_cv.groupby("Class")
    .first()
    .reset_index()
)

fig, axes = plt.subplots(3, 3, figsize=(15, 13))

for row, (_, record) in enumerate(sample_records.iterrows()):

    file_path = os.path.join(
        DATASET_PATH,
        f"part_{int(record['Part'])}",
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        image = np.array(
            f["cjdata"]["image"],
            dtype=np.float32
        )

    # Method 1: Min-max normalization
    minmax = (
        (image - image.min()) /
        (image.max() - image.min() + 1e-8)
    )

    # Method 2: Percentile clipping and normalization
    low = np.percentile(image, 0.5)
    high = np.percentile(image, 99.5)

    clipped = np.clip(image, low, high)

    percentile_norm = (
        (clipped - low) /
        (high - low + 1e-8)
    )

    axes[row, 0].imshow(image, cmap="gray")
    axes[row, 0].set_title(f"{record['Class']} - Original")

    axes[row, 1].imshow(minmax, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("Min-max normalization")

    axes[row, 2].imshow(
        percentile_norm,
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[row, 2].set_title("Percentile normalization")

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
print("IMAGE DIMENSION DISTRIBUTION")
print("=" * 50)

display(
    df_cv["Image_Shape"].value_counts().to_frame("Number of Images")
)

print("\nIMAGE DIMENSIONS BY TUMOR CLASS")
print("=" * 50)

display(
    pd.crosstab(
        df_cv["Class"],
        df_cv["Image_Shape"]
    )
)

In [ ]:
dimension_summary = (
    df_cv.assign(
        Image_Dimensions=df_cv["Image_Shape"].astype(str)
    )
    .groupby(["Class", "Image_Dimensions"])
    .size()
    .unstack(fill_value=0)
)

print("IMAGE DIMENSIONS BY TUMOR CLASS")
print("=" * 55)

display(dimension_summary)

In [ ]:
from PIL import Image
import numpy as np
import h5py
import os
import matplotlib.pyplot as plt

def preprocess_mri(image, target_size=(224, 224)):

    image = np.asarray(image, dtype=np.float32)

    # Resize while preserving the square image geometry
    pil_image = Image.fromarray(image, mode="F")

    resized = pil_image.resize(
        target_size,
        resample=Image.Resampling.BILINEAR
    )

    resized = np.asarray(resized, dtype=np.float32)

    # Per-image min-max normalization
    minimum = resized.min()
    maximum = resized.max()

    normalized = (
        (resized - minimum) /
        (maximum - minimum + 1e-8)
    )

    return normalized


# Test one image from each tumor class
test_records = df_cv.groupby("Class").first().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (_, record) in zip(axes, test_records.iterrows()):

    file_path = os.path.join(
        DATASET_PATH,
        f"part_{int(record['Part'])}",
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        original = np.array(f["cjdata"]["image"])

    processed = preprocess_mri(original)

    print(f"\nClass: {record['Class']}")
    print("Original shape:", original.shape)
    print("Processed shape:", processed.shape)
    print("Data type:", processed.dtype)
    print("Minimum:", processed.min())
    print("Maximum:", processed.max())

    ax.imshow(processed, cmap="gray", vmin=0, vmax=1)
    ax.set_title(record["Class"])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Select all 15 smaller MRI images
small_image_records = df_cv[
    df_cv["Image_Shape"].astype(str) == "(256, 256)"
]

print("SMALL IMAGE PREPROCESSING TEST")
print("=" * 55)

print("Number of smaller images:", len(small_image_records))

for _, record in small_image_records.head(3).iterrows():

    file_path = os.path.join(
        DATASET_PATH,
        f"part_{int(record['Part'])}",
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        original = np.array(f["cjdata"]["image"])

    processed = preprocess_mri(original)

    print("\nFilename:", record["Filename"])
    print("Patient ID:", record["Patient_ID"])
    print("Original shape:", original.shape)
    print("Processed shape:", processed.shape)
    print("Data type:", processed.dtype)
    print("Intensity range:",
          processed.min(), "to", processed.max())

In [ ]:
%pip install tqdm

In [ ]:
import os
import numpy as np
import h5py
from tqdm.notebook import tqdm

# Output directory
PROCESSED_PATH = os.path.join(
    r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI",
    "dataset",
    "processed"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)

# Ensure records are in numeric filename order
processing_df = df_cv.sort_values(
    "File_Number"
).reset_index(drop=True)

# Create memory-mapped array
images_path = os.path.join(
    PROCESSED_PATH,
    "brain_mri_images_224.npy"
)

images = np.lib.format.open_memmap(
    images_path,
    mode="w+",
    dtype=np.float32,
    shape=(len(processing_df), 224, 224)
)

# Process every MRI image
for i, record in tqdm(
    processing_df.iterrows(),
    total=len(processing_df),
    desc="Preprocessing MRI images"
):

    file_path = os.path.join(
        DATASET_PATH,
        f"part_{int(record['Part'])}",
        str(record["Filename"])
    )

    with h5py.File(file_path, "r") as f:
        original = np.array(
            f["cjdata"]["image"],
            dtype=np.float32
        )

    processed = preprocess_mri(original)

    images[i] = processed

# Flush processed images to disk
images.flush()

# Save corresponding labels
labels = processing_df["Label"].to_numpy(dtype=np.int64)

labels_path = os.path.join(
    PROCESSED_PATH,
    "brain_mri_labels.npy"
)

np.save(labels_path, labels)

# Save matching metadata
metadata_path = os.path.join(
    PROCESSED_PATH,
    "brain_mri_metadata.csv"
)

processing_df.to_csv(metadata_path, index=False)

print("\nCOMPLETE DATASET PREPROCESSING")
print("=" * 55)

print("Total processed images:", len(images))
print("Image array shape:", images.shape)
print("Data type:", images.dtype)
print("Labels shape:", labels.shape)
print("Unique patients:", processing_df["Patient_ID"].nunique())
print("Unique folds:", processing_df["Fold"].nunique())

print("\nSaved files:")
print(images_path)
print(labels_path)
print(metadata_path)

In [ ]:
import numpy as np
import pandas as pd
import os

PROCESSED_PATH = (
    r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
    r"\dataset\processed"
)

# Reload saved files
X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

print("SAVED DATASET INTEGRITY CHECK")
print("=" * 55)

print("Images shape:", X.shape)
print("Labels shape:", y.shape)
print("Metadata shape:", metadata.shape)

print("\nImage data type:", X.dtype)
print("Image intensity range:", X.min(), "to", X.max())

print("\nLabel distribution:")
print(pd.Series(y).value_counts().sort_index())

print("\nPatient count:", metadata["Patient_ID"].nunique())
print("Fold count:", metadata["Fold"].nunique())

print("\nMissing values in metadata:", metadata.isnull().sum().sum())

print("\nImage-label-metadata alignment:",
      len(X) == len(y) == len(metadata))

In [ ]:
import importlib.util
import platform
import os
import subprocess

print("COMPUTER TRAINING ENVIRONMENT")
print("=" * 55)

print("Operating system:", platform.platform())
print("Python version:", platform.python_version())
print("CPU logical cores:", os.cpu_count())

print("\nDEEP LEARNING FRAMEWORKS")
print("=" * 55)

print("PyTorch installed:",
      importlib.util.find_spec("torch") is not None)

print("TensorFlow installed:",
      importlib.util.find_spec("tensorflow") is not None)

print("\nNVIDIA GPU CHECK")
print("=" * 55)

try:
    result = subprocess.run(
        ["nvidia-smi"],
        capture_output=True,
        text=True,
        timeout=10
    )

    if result.returncode == 0:
        print(result.stdout)
    else:
        print("NVIDIA GPU information unavailable.")
        print(result.stderr)

except Exception:
    print("NVIDIA GPU not detected through nvidia-smi.")

In [ ]:
import torch
import torchvision

print("DEEP LEARNING ENVIRONMENT")
print("=" * 45)

print("PyTorch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Selected device:", device)

In [ ]:
import psutil

memory = psutil.virtual_memory()

print("SYSTEM MEMORY")
print("=" * 45)
print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"Used RAM: {memory.used / (1024**3):.2f} GB")
print(f"RAM Usage: {memory.percent}%")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class BrainMRIDataset(Dataset):

    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):

        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = int(self.labels[actual_index]) - 1
        label = torch.tensor(label, dtype=torch.long)

        return image, label


print("PyTorch Dataset class created successfully.")
print("Input image shape: (1, 224, 224)")
print("Output classes: 0, 1, 2")

In [ ]:
from torch.utils.data import DataLoader
import numpy as np

# Patient-independent fold split
train_indices = np.where(metadata["Fold"].to_numpy() != 1)[0]
val_indices = np.where(metadata["Fold"].to_numpy() == 1)[0]

# Create datasets
train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("PILOT DATA LOADERS")
print("=" * 45)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Training patients:", metadata.iloc[train_indices]["Patient_ID"].nunique())
print("Validation patients:", metadata.iloc[val_indices]["Patient_ID"].nunique())

In [ ]:
import torch.nn as nn

class BrainTumorCNN(nn.Module):

    def __init__(self, num_classes=3):
        super(BrainTumorCNN, self).__init__()

        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            # Block 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),

            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),

            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BrainTumorCNN()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("BRAIN TUMOR CNN")
print("=" * 45)
print(model)
print("\nTotal parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")

In [ ]:
# Select one batch
images, labels = next(iter(train_loader))

# Forward pass
model.eval()

with torch.no_grad():
    outputs = model(images)

print("CNN FORWARD PASS TEST")
print("=" * 45)

print("Input batch shape:", images.shape)
print("Labels shape:", labels.shape)
print("Output shape:", outputs.shape)

print("\nPredicted class indices:", outputs.argmax(dim=1).tolist())
print("Actual class indices:", labels.tolist())

print("\nForward pass successful!")

In [ ]:
import torch.nn as nn
import torch.optim as optim

device = torch.device("cpu")

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("TRAINING COMPONENTS")
print("=" * 45)
print("Device:", device)
print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

In [ ]:
import time
import psutil

model.train()

pilot_batches = 20
total_loss = 0.0

start_time = time.time()

for batch_index, (images, labels) in enumerate(train_loader):

    if batch_index >= pilot_batches:
        break

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

elapsed_time = time.time() - start_time

memory = psutil.virtual_memory()

print("CNN TRAINING PILOT")
print("=" * 45)
print("Batches completed:", pilot_batches)
print("Average training loss:", round(total_loss / pilot_batches, 4))
print("Total time:", round(elapsed_time, 2), "seconds")
print("Average time per batch:", round(elapsed_time / pilot_batches, 2), "seconds")
print("Available RAM:", round(memory.available / (1024**3), 2), "GB")
print("RAM usage:", memory.percent, "%")

In [ ]:
import os
import torch

MODEL_DIR = os.path.join(PROJECT_PATH, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

pilot_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_pilot.pth"
)

torch.save(model.state_dict(), pilot_path)

print("PILOT CHECKPOINT")
print("=" * 45)
print("Saved:", pilot_path)
print("File exists:", os.path.exists(pilot_path))
print("File size:", round(os.path.getsize(pilot_path) / 1024, 2), "KB")

In [ ]:
import time
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    start_time = time.time()

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    average_loss = total_loss / len(loader)

    training_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    elapsed_time = time.time() - start_time

    return average_loss, training_accuracy, elapsed_time


def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    average_loss = total_loss / len(loader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "confusion_matrix": confusion_matrix(
            all_labels,
            all_predictions
        ),
        "classification_report": classification_report(
            all_labels,
            all_predictions,
            target_names=[
                "Meningioma",
                "Glioma",
                "Pituitary"
            ],
            zero_division=0
        )
    }


print("Training and evaluation functions created successfully.")

In [ ]:
# Complete one training epoch
train_loss, train_accuracy, epoch_time = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device
)

# Evaluate on held-out Fold 1
validation_results = evaluate_model(
    model,
    val_loader,
    criterion,
    device
)

print("FIRST CNN TRAINING EPOCH")
print("=" * 50)

print(f"Training Loss:       {train_loss:.4f}")
print(f"Training Accuracy:   {train_accuracy:.4f}")
print(f"Validation Loss:     {validation_results['loss']:.4f}")
print(f"Validation Accuracy: {validation_results['accuracy']:.4f}")
print(f"Balanced Accuracy:   {validation_results['balanced_accuracy']:.4f}")
print(f"Macro F1:            {validation_results['macro_f1']:.4f}")

print(f"\nTraining Time: {epoch_time:.2f} seconds")

print("\nAvailable RAM:",
      f"{psutil.virtual_memory().available / (1024**3):.2f} GB")

In [ ]:
print("VALIDATION CONFUSION MATRIX")
print("=" * 45)

print("Rows = Actual | Columns = Predicted")
print(validation_results["confusion_matrix"])

print("\nCLASSIFICATION REPORT")
print("=" * 45)

print(validation_results["classification_report"])

print("\nCURRENT MEMORY")
print("=" * 45)

memory = psutil.virtual_memory()

print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM usage: {memory.percent}%")

In [ ]:
import os
import torch
import gc
import psutil

checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_fold1_epoch1.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 1,
    "epoch": 1,
    "validation_accuracy": validation_results["accuracy"],
    "balanced_accuracy": validation_results["balanced_accuracy"],
    "macro_f1": validation_results["macro_f1"]
}, checkpoint_path)

gc.collect()

print("FIRST EPOCH CHECKPOINT SAVED")
print("=" * 45)
print("File:", checkpoint_path)
print("Exists:", os.path.exists(checkpoint_path))
print("Size:", round(os.path.getsize(checkpoint_path) / 1024, 2), "KB")
print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import psutil

PROJECT_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "dataset", "processed"
)

MODEL_DIR = os.path.join(PROJECT_PATH, "models")

X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

device = torch.device("cpu")

print("DATASET RESTORED")
print("=" * 45)
print("Images:", X.shape)
print("Labels:", y.shape)
print("Patients:", metadata["Patient_ID"].nunique())
print("Folds:", metadata["Fold"].nunique())
print("Device:", device)
print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = int(self.labels[actual_index]) - 1
        label = torch.tensor(label, dtype=torch.long)

        return image, label


class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(BrainTumorCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = BrainTumorCNN().to(device)

print("CNN ARCHITECTURE RESTORED")
print("=" * 45)
print("Total parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", device)

In [ ]:
train_indices = np.where(
    metadata["Fold"].to_numpy() != 1
)[0]

val_indices = np.where(
    metadata["Fold"].to_numpy() == 1
)[0]

train_dataset = BrainMRIDataset(
    X, y, train_indices
)

val_dataset = BrainMRIDataset(
    X, y, val_indices
)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print("PATIENT-INDEPENDENT DATA SPLIT")
print("=" * 45)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training patients:",
      metadata.iloc[train_indices]["Patient_ID"].nunique())
print("Validation patients:",
      metadata.iloc[val_indices]["Patient_ID"].nunique())

print("\nTraining class counts:")
print(pd.Series(y[train_indices]).value_counts().sort_index())

print("\nValidation class counts:")
print(pd.Series(y[val_indices]).value_counts().sort_index())

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1, 2])

train_labels_zero = y[train_indices] - 1

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels_zero
)

class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=device
)

print("CLASS WEIGHT CALCULATION")
print("=" * 45)

class_names = [
    "Meningioma",
    "Glioma",
    "Pituitary"
]

for name, weight in zip(class_names, weights):
    print(f"{name}: {weight:.4f}")

print("\nWeight tensor:", class_weights)

In [ ]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("WEIGHTED TRAINING CONFIGURATION")
print("=" * 45)
print("Model:", model.__class__.__name__)
print("Loss function:", criterion.__class__.__name__)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Batch size:", train_loader.batch_size)
print("Device:", device)

memory = psutil.virtual_memory()
print(
    "Available RAM:",
    round(memory.available / (1024**3), 2),
    "GB"
)

In [ ]:
import time

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    start_time = time.time()

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    average_loss = total_loss / len(loader)

    training_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    elapsed_time = time.time() - start_time

    return average_loss, training_accuracy, elapsed_time


def evaluate_model(model, loader, criterion, device):

    model.eval()

    total_loss = 0.0
    all_predictions = []
    all_labels = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    average_loss = total_loss / len(loader)

    return {
        "loss": average_loss,
        "accuracy": accuracy_score(all_labels, all_predictions),
        "balanced_accuracy": balanced_accuracy_score(
            all_labels, all_predictions
        ),
        "macro_f1": f1_score(
            all_labels, all_predictions, average="macro"
        ),
        "confusion_matrix": confusion_matrix(
            all_labels, all_predictions
        ),
        "classification_report": classification_report(
            all_labels,
            all_predictions,
            labels=[0, 1, 2],
            target_names=[
                "Meningioma",
                "Glioma",
                "Pituitary"
            ],
            zero_division=0
        )
    }


print("TRAINING AND EVALUATION FUNCTIONS RESTORED")
print("=" * 45)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

In [ ]:
import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"

PROCESSED_PATH = os.path.join(PROJECT_PATH, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_PATH, "models")

X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

device = torch.device("cpu")

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label


class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


train_indices = np.where(metadata["Fold"].to_numpy() != 1)[0]
val_indices = np.where(metadata["Fold"].to_numpy() == 1)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

model = BrainTumorCNN().to(device)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=y[train_indices] - 1
)

class_weights = torch.tensor(
    weights, dtype=torch.float32, device=device
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001
)

print("TRAINING ENVIRONMENT RESTORED")
print("=" * 45)
print("Images:", X.shape)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training patients:",
      metadata.iloc[train_indices]["Patient_ID"].nunique())
print("Validation patients:",
      metadata.iloc[val_indices]["Patient_ID"].nunique())
print("Batch size:", train_loader.batch_size)
print("Class weights:", class_weights.tolist())
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

In [ ]:

import os
import gc
import time
import psutil
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

torch.set_num_threads(4)

# ==========================================
# TRAINING — EPOCH 1
# ==========================================

model.train()

total_loss = 0.0
train_predictions = []
train_labels = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_labels.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(
            f"Training batch {batch_number}/{len(train_loader)}"
        )

train_loss = total_loss / len(train_loader)

train_accuracy = accuracy_score(
    train_labels, train_predictions
)

training_time = time.time() - start_time

# ==========================================
# VALIDATION
# ==========================================

model.eval()

validation_loss = 0.0
val_predictions = []
val_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(
    val_labels, val_predictions
)

val_balanced_accuracy = balanced_accuracy_score(
    val_labels, val_predictions
)

val_macro_f1 = f1_score(
    val_labels, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_labels, val_predictions, labels=[0, 1, 2]
)

# ==========================================
# SAVE CHECKPOINT
# ==========================================

checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch1.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 1,
    "epoch": 1,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": weights.tolist()
}, checkpoint_path)

gc.collect()

# ==========================================
# RESULTS
# ==========================================

print("\nCLASS-WEIGHTED CNN — EPOCH 1")
print("=" * 55)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_labels,
    val_predictions,
    labels=[0, 1, 2],
    target_names=[
        "Meningioma",
        "Glioma",
        "Pituitary"
    ],
    zero_division=0
))

print("\nCHECKPOINT SAVED")
print(checkpoint_path)

memory = psutil.virtual_memory()
print("\nAvailable RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader

PROJECT_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
PROCESSED_PATH = os.path.join(PROJECT_PATH, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_PATH, "models")

# Restore dataset
X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

device = torch.device("cpu")

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label


# CNN architecture
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Restore patient-independent split
train_indices = np.where(
    metadata["Fold"].to_numpy() != 1
)[0]

val_indices = np.where(
    metadata["Fold"].to_numpy() == 1
)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Restore checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch1.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model = BrainTumorCNN().to(device)

model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

class_weights = torch.tensor(
    checkpoint["class_weights"],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

torch.set_num_threads(4)
gc.collect()

print("CHECKPOINT RESTORED")
print("=" * 45)
print("Previous epoch:", checkpoint["epoch"])
print("Previous validation accuracy:",
      checkpoint["validation_accuracy"])
print("Previous balanced accuracy:",
      checkpoint["balanced_accuracy"])
print("Previous macro F1:",
      checkpoint["macro_f1"])
print("Training batch size:", train_loader.batch_size)
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")


In [ ]:

import time
import gc
import psutil
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.train()

total_loss = 0.0
train_predictions = []
train_labels = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_labels.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(f"Training batch {batch_number}/{len(train_loader)}")

train_loss = total_loss / len(train_loader)
train_accuracy = accuracy_score(train_labels, train_predictions)
training_time = time.time() - start_time

# Validation
model.eval()

validation_loss = 0.0
val_predictions = []
val_labels = []

with torch.no_grad():
    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(val_labels, val_predictions)

val_balanced_accuracy = balanced_accuracy_score(
    val_labels, val_predictions
)

val_macro_f1 = f1_score(
    val_labels, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_labels, val_predictions, labels=[0, 1, 2]
)

# Save epoch 2 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch2.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 1,
    "epoch": 2,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.cpu().tolist()
}, checkpoint_path)

gc.collect()

print("\nCLASS-WEIGHTED CNN — EPOCH 2")
print("=" * 55)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_labels,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nCheckpoint saved:", checkpoint_path)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader

PROJECT_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
PROCESSED_PATH = os.path.join(PROJECT_PATH, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_PATH, "models")

X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)
y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)
metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

device = torch.device("cpu")

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label


class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


train_indices = np.where(metadata["Fold"].to_numpy() != 1)[0]
val_indices = np.where(metadata["Fold"].to_numpy() == 1)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset, batch_size=4, shuffle=True,
    num_workers=0, pin_memory=False
)
val_loader = DataLoader(
    val_dataset, batch_size=4, shuffle=False,
    num_workers=0, pin_memory=False
)

checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_weighted_fold1_epoch2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model = BrainTumorCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

class_weights = torch.tensor(
    checkpoint["class_weights"],
    dtype=torch.float32,
    device=device
)
criterion = nn.CrossEntropyLoss(weight=class_weights)

torch.set_num_threads(4)
gc.collect()

print("EPOCH 2 CHECKPOINT RESTORED")
print("=" * 45)
print("Loaded epoch:", checkpoint["epoch"])
print("Macro F1:", checkpoint["macro_f1"])
print("Balanced accuracy:", checkpoint["balanced_accuracy"])
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Batch size:", train_loader.batch_size)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")


In [ ]:

import time
import gc
import psutil
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# TRAINING — EPOCH 3
# ==========================================

model.train()

total_loss = 0.0
train_predictions = []
train_labels = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_labels.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(f"Training batch {batch_number}/{len(train_loader)}")

train_loss = total_loss / len(train_loader)
train_accuracy = accuracy_score(train_labels, train_predictions)
training_time = time.time() - start_time

# ==========================================
# VALIDATION
# ==========================================

model.eval()

validation_loss = 0.0
val_predictions = []
val_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(val_labels, val_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    val_labels, val_predictions
)
val_macro_f1 = f1_score(
    val_labels, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_labels, val_predictions, labels=[0, 1, 2]
)

# ==========================================
# SAVE EPOCH 3
# ==========================================

checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch3.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 1,
    "epoch": 3,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.cpu().tolist()
}, checkpoint_path)

gc.collect()

# ==========================================
# RESULTS
# ==========================================

print("\nCLASS-WEIGHTED CNN — EPOCH 3")
print("=" * 55)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_labels,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nCheckpoint saved:", checkpoint_path)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader

PROJECT_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
PROCESSED_PATH = os.path.join(PROJECT_PATH, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_PATH, "models")

# Restore dataset
X = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_images_224.npy"),
    mmap_mode="r"
)
y = np.load(
    os.path.join(PROCESSED_PATH, "brain_mri_labels.npy")
)
metadata = pd.read_csv(
    os.path.join(PROCESSED_PATH, "brain_mri_metadata.csv")
)

device = torch.device("cpu")

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label


# CNN
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Restore original patient-independent split
train_indices = np.where(
    metadata["Fold"].to_numpy() != 1
)[0]

val_indices = np.where(
    metadata["Fold"].to_numpy() == 1
)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Restore best checkpoint: Epoch 2
checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model = BrainTumorCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

# Lower learning rate
for param_group in optimizer.param_groups:
    param_group["lr"] = 0.0003

class_weights = torch.tensor(
    checkpoint["class_weights"],
    dtype=torch.float32,
    device=device
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

torch.set_num_threads(4)
gc.collect()

print("BEST CHECKPOINT RESTORED")
print("=" * 45)
print("Starting epoch:", checkpoint["epoch"])
print("Starting macro F1:", round(checkpoint["macro_f1"], 4))
print("Starting balanced accuracy:",
      round(checkpoint["balanced_accuracy"], 4))
print("New learning rate:", optimizer.param_groups[0]["lr"])
print("Batch size:", train_loader.batch_size)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")


In [ ]:

import time
import gc
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Training
model.train()

total_loss = 0.0
train_predictions = []
train_labels = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_labels.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(f"Training batch {batch_number}/{len(train_loader)}")

train_loss = total_loss / len(train_loader)
train_accuracy = accuracy_score(train_labels, train_predictions)
training_time = time.time() - start_time

# Validation
model.eval()

validation_loss = 0.0
val_predictions = []
val_labels = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_labels.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(val_labels, val_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    val_labels, val_predictions
)
val_macro_f1 = f1_score(
    val_labels, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_labels, val_predictions, labels=[0, 1, 2]
)

# Save separate experiment checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_lr0003_fold1_epoch3.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 1,
    "epoch": 3,
    "learning_rate": 0.0003,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.cpu().tolist()
}, checkpoint_path)

gc.collect()

# Results
print("\nREDUCED LEARNING RATE — EPOCH 3")
print("=" * 55)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_labels,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nCheckpoint saved:", checkpoint_path)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import torch
import numpy as np
import pandas as pd
import psutil

from torch.utils.data import Dataset, DataLoader

# Paths
PROJECT_DIR = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
DATA_DIR = os.path.join(PROJECT_DIR, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")

X_PATH = os.path.join(DATA_DIR, "brain_mri_images_224.npy")
Y_PATH = os.path.join(DATA_DIR, "brain_mri_labels.npy")
METADATA_PATH = os.path.join(DATA_DIR, "brain_mri_metadata.csv")

BEST_CHECKPOINT = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_weighted_fold1_epoch2.pth"
)

# Load dataset using memory mapping
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata = pd.read_csv(METADATA_PATH)

# Dataset class
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label

# Model architecture
class BrainTumorCNN(torch.nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, padding=1),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(16, 32, 3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(2),

            torch.nn.Conv2d(32, 64, 3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(inplace=True),
            torch.nn.MaxPool2d(2)
        )

        self.classifier = torch.nn.Sequential(
            torch.nn.AdaptiveAvgPool2d((1, 1)),
            torch.nn.Flatten(),
            torch.nn.Linear(64, 32),
            torch.nn.ReLU(inplace=True),
            torch.nn.Dropout(0.4),
            torch.nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Device and training configuration
device = torch.device("cpu")
torch.set_num_threads(4)

BATCH_SIZE = 4

# Recreate Fold 1 splits
train_indices = np.where(
    metadata["Fold"].values != 1
)[0]

val_indices = np.where(
    metadata["Fold"].values == 1
)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Restore best checkpoint
checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model = BrainTumorCNN().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

criterion = torch.nn.CrossEntropyLoss(
    weight=torch.tensor(
        checkpoint["class_weights"],
        dtype=torch.float32
    )
)

gc.collect()

print("BEST CHECKPOINT RESTORED")
print("=" * 45)
print("Checkpoint:", os.path.basename(BEST_CHECKPOINT))
print("Epoch:", checkpoint["epoch"])
print("Macro F1:", checkpoint["macro_f1"])
print("Balanced accuracy:", checkpoint["balanced_accuracy"])
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Batch size:", BATCH_SIZE)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import gc
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

model.eval()

all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch_number, (images, labels) in enumerate(val_loader, 1):

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_true_labels.extend(labels.numpy())

        if batch_number % 50 == 0:
            print(f"Inference batch {batch_number}/{len(val_loader)}")

# Convert labels back to original dataset labels: 1, 2, 3
all_predictions = np.array(all_predictions) + 1
all_true_labels = np.array(all_true_labels) + 1

# Save validation predictions with patient metadata
prediction_results = metadata.iloc[val_indices].copy().reset_index(drop=True)

prediction_results["Actual_Label"] = all_true_labels
prediction_results["Predicted_Label"] = all_predictions

prediction_path = os.path.join(
    PROJECT_DIR,
    "results",
    "fold1_best_epoch2_predictions.csv"
)

prediction_results.to_csv(prediction_path, index=False)

print("\nFOLD 1 BEST MODEL — PREDICTION AUDIT")
print("=" * 50)

print("Accuracy:",
      round(accuracy_score(all_true_labels, all_predictions), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(all_true_labels, all_predictions), 4))

print("Macro F1:",
      round(f1_score(all_true_labels, all_predictions, average="macro"), 4))

print("\nConfusion matrix:")
print(confusion_matrix(all_true_labels, all_predictions, labels=[1, 2, 3]))

print("\nClassification report:")
print(classification_report(
    all_true_labels,
    all_predictions,
    labels=[1, 2, 3],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nPredictions saved:", prediction_path)

gc.collect()
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")


In [ ]:

# Patient-level error analysis for Fold 1

prediction_results["Correct"] = (
    prediction_results["Actual_Label"]
    == prediction_results["Predicted_Label"]
)

patient_errors = (
    prediction_results
    .groupby(["Patient_ID", "Class"], observed=True)
    .agg(
        Total_Images=("Correct", "size"),
        Correct_Images=("Correct", "sum")
    )
    .reset_index()
)

patient_errors["Incorrect_Images"] = (
    patient_errors["Total_Images"]
    - patient_errors["Correct_Images"]
)

patient_errors["Accuracy"] = (
    patient_errors["Correct_Images"]
    / patient_errors["Total_Images"]
)

print("PATIENT-LEVEL VALIDATION ERROR ANALYSIS")
print("=" * 55)

print("Validation patients:", patient_errors["Patient_ID"].nunique())

print("\nPatient-level accuracy summary:")
print(
    patient_errors.groupby("Class", observed=True)["Accuracy"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
)

print("\nPatients with lowest accuracy:")
print(
    patient_errors.sort_values("Accuracy")
    .head(15)
    .to_string(index=False)
)

print("\nTotal incorrectly classified images:",
      int(patient_errors["Incorrect_Images"].sum()))

analysis_path = os.path.join(
    PROJECT_DIR,
    "results",
    "fold1_patient_error_analysis.csv"
)

patient_errors.to_csv(analysis_path, index=False)

print("\nSaved:", analysis_path)


In [ ]:

# Focus on the three Glioma patients with zero correct predictions

difficult_patients = [101384, 108479, 93825]

for patient_id in difficult_patients:

    patient_data = prediction_results[
        prediction_results["Patient_ID"].astype(str)
        == str(patient_id)
    ]

    print(f"\nPATIENT: {patient_id}")
    print("-" * 45)

    print("Actual class:", patient_data["Class"].iloc[0])
    print("Total MRI images:", len(patient_data))

    print("\nPredicted class distribution:")

    predicted_counts = (
        patient_data["Predicted_Label"]
        .map({
            1: "Meningioma",
            2: "Glioma",
            3: "Pituitary"
        })
        .value_counts()
    )

    print(predicted_counts.to_string())

    print("\nIndividual predictions:")
    print(
        patient_data[
            ["Patient_ID", "Actual_Label", "Predicted_Label", "Correct"]
        ].to_string(index=False)
    )

print("\n\nOVERALL GLIOMA MISCLASSIFICATION PATTERN")
print("=" * 50)

glioma_errors = prediction_results[
    (prediction_results["Actual_Label"] == 2)
    & (prediction_results["Predicted_Label"] != 2)
]

print("Misclassified Glioma images:", len(glioma_errors))

print("\nWhere were they classified?")
print(
    glioma_errors["Predicted_Label"]
    .map({
        1: "Meningioma",
        2: "Glioma",
        3: "Pituitary"
    })
    .value_counts()
    .to_string()
)


In [ ]:

import os
import gc
import time
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Paths
PROJECT_DIR = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
DATA_DIR = os.path.join(PROJECT_DIR, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data
X = np.load(
    os.path.join(DATA_DIR, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(DATA_DIR, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(DATA_DIR, "brain_mri_metadata.csv")
)

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label

# CNN
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Configuration
FOLD = 2
BATCH_SIZE = 4
LEARNING_RATE = 0.001
EPOCHS = 2

device = torch.device("cpu")
torch.set_num_threads(4)

# Patient-independent split
train_indices = np.where(
    metadata["Fold"].values != FOLD
)[0]

val_indices = np.where(
    metadata["Fold"].values == FOLD
)[0]

train_patients = set(
    metadata.iloc[train_indices]["Patient_ID"].astype(str)
)

val_patients = set(
    metadata.iloc[val_indices]["Patient_ID"].astype(str)
)

assert train_patients.isdisjoint(val_patients), \
    "Patient leakage detected!"

# Loaders
train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Calculate class weights from training data only
train_labels = y[train_indices] - 1

class_counts = np.bincount(
    train_labels,
    minlength=3
)

class_weights = len(train_labels) / (
    3 * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
)

# Fresh model
model = BrainTumorCNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# Fold 2 checkpoint destination
CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_fold2_best.pth"
)

gc.collect()

print("FOLD 2 — TRAINING ENVIRONMENT READY")
print("=" * 50)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Patient overlap:", len(train_patients & val_patients))
print("Training class counts:", class_counts)
print("Class weights:", class_weights.tolist())
print("Architecture parameters:",
      sum(p.numel() for p in model.parameters()))
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("Fresh model initialization: YES")

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import time
import gc
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.train()

total_loss = 0.0
train_predictions = []
train_true = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_true.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(f"Training batch {batch_number}/{len(train_loader)}")

train_loss = total_loss / len(train_loader)
train_accuracy = accuracy_score(train_true, train_predictions)

training_time = time.time() - start_time

# Validation
model.eval()

validation_loss = 0.0
val_predictions = []
val_true = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_true.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(val_true, val_predictions)

val_balanced_accuracy = balanced_accuracy_score(
    val_true, val_predictions
)

val_macro_f1 = f1_score(
    val_true, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_true, val_predictions, labels=[0, 1, 2]
)

# Save Epoch 1
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 2,
    "epoch": 1,
    "learning_rate": LEARNING_RATE,
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, CHECKPOINT_PATH)

gc.collect()

print("\nFOLD 2 — EPOCH 1 RESULTS")
print("=" * 50)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_true,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nCheckpoint saved:", CHECKPOINT_PATH)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

PROJECT_DIR = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
DATA_DIR = os.path.join(PROJECT_DIR, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")

# Load dataset
X = np.load(
    os.path.join(DATA_DIR, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(DATA_DIR, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(DATA_DIR, "brain_mri_metadata.csv")
)

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label

# CNN
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Configuration
FOLD = 2
BATCH_SIZE = 4
device = torch.device("cpu")

torch.set_num_threads(4)

# Recreate splits
train_indices = np.where(
    metadata["Fold"].values != FOLD
)[0]

val_indices = np.where(
    metadata["Fold"].values == FOLD
)[0]

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Restore checkpoint
CHECKPOINT_PATH = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_fold2_best.pth"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model = BrainTumorCNN().to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

class_weights = torch.tensor(
    checkpoint["class_weights"],
    dtype=torch.float32
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

gc.collect()

print("FOLD 2 CHECKPOINT RESTORED")
print("=" * 45)
print("Starting epoch:", checkpoint["epoch"])
print("Starting macro F1:", checkpoint["macro_f1"])
print("Starting balanced accuracy:", checkpoint["balanced_accuracy"])
print("Learning rate:",
      optimizer.param_groups[0]["lr"])
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Batch size:", BATCH_SIZE)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import time
import gc
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.train()

total_loss = 0.0
train_predictions = []
train_true = []

start_time = time.time()

for batch_number, (images, labels) in enumerate(train_loader, 1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    predictions = outputs.argmax(dim=1)

    train_predictions.extend(predictions.cpu().numpy())
    train_true.extend(labels.cpu().numpy())

    if batch_number % 100 == 0:
        print(f"Training batch {batch_number}/{len(train_loader)}")

train_loss = total_loss / len(train_loader)
train_accuracy = accuracy_score(train_true, train_predictions)

training_time = time.time() - start_time

# Validation
model.eval()

validation_loss = 0.0
val_predictions = []
val_true = []

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        validation_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        val_predictions.extend(predictions.cpu().numpy())
        val_true.extend(labels.cpu().numpy())

validation_loss /= len(val_loader)

val_accuracy = accuracy_score(val_true, val_predictions)

val_balanced_accuracy = balanced_accuracy_score(
    val_true, val_predictions
)

val_macro_f1 = f1_score(
    val_true, val_predictions, average="macro"
)

val_confusion = confusion_matrix(
    val_true, val_predictions, labels=[0, 1, 2]
)

# Save Epoch 2 checkpoint separately
EPOCH2_PATH = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_fold2_epoch2.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 2,
    "epoch": 2,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": validation_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, EPOCH2_PATH)

gc.collect()

print("\nFOLD 2 — EPOCH 2 RESULTS")
print("=" * 50)

print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {validation_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {training_time:.2f} seconds")

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted")
print(val_confusion)

print("\nCLASSIFICATION REPORT")
print(classification_report(
    val_true,
    val_predictions,
    labels=[0, 1, 2],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nCheckpoint saved:", EPOCH2_PATH)

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:

import os
import gc
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
DATA_DIR = os.path.join(PROJECT_DIR, "dataset", "processed")
MODEL_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

X = np.load(
    os.path.join(DATA_DIR, "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(DATA_DIR, "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(DATA_DIR, "brain_mri_metadata.csv")
)

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label

class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Restore Fold 2 best checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR,
    "brain_tumor_cnn_fold2_epoch2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

model = BrainTumorCNN()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Fold 2 validation split
val_indices = np.where(
    metadata["Fold"].values == 2
)[0]

val_dataset = BrainMRIDataset(X, y, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

all_predictions = []
all_true = []

with torch.no_grad():

    for batch_number, (images, labels) in enumerate(val_loader, 1):

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.numpy())
        all_true.extend(labels.numpy())

        if batch_number % 50 == 0:
            print(f"Inference batch {batch_number}/{len(val_loader)}")

# Restore original dataset labels
all_predictions = np.array(all_predictions) + 1
all_true = np.array(all_true) + 1

# Preserve patient metadata and predictions
fold2_results = metadata.iloc[val_indices].copy().reset_index(drop=True)

fold2_results["Actual_Label"] = all_true
fold2_results["Predicted_Label"] = all_predictions
fold2_results["Correct"] = (
    fold2_results["Actual_Label"]
    == fold2_results["Predicted_Label"]
)

prediction_path = os.path.join(
    RESULTS_DIR,
    "fold2_best_epoch2_predictions.csv"
)

fold2_results.to_csv(prediction_path, index=False)

print("\nFOLD 2 — FINAL PREDICTION AUDIT")
print("=" * 50)

print("Accuracy:",
      round(accuracy_score(all_true, all_predictions), 4))

print("Balanced accuracy:",
      round(balanced_accuracy_score(all_true, all_predictions), 4))

print("Macro F1:",
      round(f1_score(all_true, all_predictions, average="macro"), 4))

print("\nConfusion matrix:")
print(confusion_matrix(all_true, all_predictions, labels=[1, 2, 3]))

print("\nClassification report:")
print(classification_report(
    all_true,
    all_predictions,
    labels=[1, 2, 3],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    zero_division=0
))

print("\nPredictions saved:", prediction_path)

gc.collect()

memory = psutil.virtual_memory()
print("Available RAM:",
      round(memory.available / (1024**3), 2), "GB")


In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
import psutil

# Fold 3 split
train_indices = np.where(metadata["Fold"].values != 3)[0]
val_indices = np.where(metadata["Fold"].values == 3)[0]

# Verify patient independence
train_patients = set(
    metadata.iloc[train_indices]["Patient_ID"].astype(str)
)

val_patients = set(
    metadata.iloc[val_indices]["Patient_ID"].astype(str)
)

overlap = train_patients.intersection(val_patients)

# Training-only class weights
train_labels = y[train_indices]
classes = np.array([1, 2, 3])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
)

print("FOLD 3 DATASET VERIFICATION")
print("=" * 45)
print("Training images:", len(train_indices))
print("Validation images:", len(val_indices))
print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Patient overlap:", len(overlap))

print("\nTraining class counts:")
print(
    pd.Series(train_labels)
    .value_counts()
    .sort_index()
)

print("\nClass weights:", class_weights)

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

assert len(overlap) == 0, "Patient leakage detected!"
assert len(train_indices) == 2492
assert len(val_indices) == 572

print("\nSUCCESS: Fold 3 split verified.")

In [ ]:
import os
import gc
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

gc.collect()

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label


# CNN architecture
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Data loaders
BATCH_SIZE = 4

train_dataset = BrainMRIDataset(X, y, train_indices)
val_dataset = BrainMRIDataset(X, y, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Fresh model
model = BrainTumorCNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor.to(device)
)

# Checkpoint directory
MODEL_DIR = os.path.join(BASE_PATH, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

print("FOLD 3 TRAINING PIPELINE")
print("=" * 45)
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Batch size:", BATCH_SIZE)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Epochs planned: 2")
print("Device:", device)
print("Available RAM:",
      round(__import__("psutil").virtual_memory().available / (1024**3), 2),
      "GB")

assert sum(p.numel() for p in model.parameters()) == 25699

In [ ]:
import time
import gc
import psutil
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

gc.collect()

model.train()
running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

# Validation
model.eval()
val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

# Save Epoch 1 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold3_epoch1.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 3,
    "epoch": 1,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 3 — EPOCH 1 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader

gc.collect()

BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"

X_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_images_224.npy"
)
Y_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_labels.npy"
)
METADATA_PATH = os.path.join(
    BASE_PATH, "results", "verified_dataset_metadata.csv"
)
MODEL_DIR = os.path.join(BASE_PATH, "models")

# Reload data
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata = pd.read_csv(METADATA_PATH)

device = torch.device("cpu")
torch.set_num_threads(4)

train_indices = np.where(metadata["Fold"].values != 3)[0]
val_indices = np.where(metadata["Fold"].values == 3)[0]

class_weights = np.array(
    [1.45987112, 0.69511855, 1.14102564],
    dtype=np.float32
)

class_weights_tensor = torch.tensor(
    class_weights, dtype=torch.float32
)

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label


class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Recreate loaders
train_loader = DataLoader(
    BrainMRIDataset(X, y, train_indices),
    batch_size=4,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    BrainMRIDataset(X, y, val_indices),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Recreate model and optimizer
model = BrainTumorCNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

# Restore Epoch 1 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold3_epoch1.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

assert checkpoint["fold"] == 3
assert checkpoint["epoch"] == 1

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print("FOLD 3 EPOCH 1 CHECKPOINT RESTORED")
print("=" * 45)
print("Restored epoch:", checkpoint["epoch"])
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["validation_loss"])
print("Validation accuracy:", checkpoint["validation_accuracy"])
print("Balanced accuracy:", checkpoint["balanced_accuracy"])
print("Macro F1:", checkpoint["macro_f1"])
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import time
import gc
import os
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

gc.collect()

model.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

# Validation for monitoring
model.eval()

val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_loader.dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

# Save predetermined Epoch 2 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold3_epoch2.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 3,
    "epoch": 2,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 3 — EPOCH 2 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

gc.collect()

BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
RESULTS_DIR = os.path.join(BASE_PATH, "results")
MODEL_DIR = os.path.join(BASE_PATH, "models")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data
X = np.load(
    os.path.join(BASE_PATH, "dataset", "processed",
                 "brain_mri_images_224.npy"),
    mmap_mode="r"
)

y = np.load(
    os.path.join(BASE_PATH, "dataset", "processed",
                 "brain_mri_labels.npy")
)

metadata = pd.read_csv(
    os.path.join(RESULTS_DIR, "verified_dataset_metadata.csv")
)

device = torch.device("cpu")
torch.set_num_threads(4)

val_indices = np.where(metadata["Fold"].values == 3)[0]

# Dataset
class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]
        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )
        image = torch.from_numpy(image).unsqueeze(0)
        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )
        return image, label


# Model
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


val_loader = DataLoader(
    BrainMRIDataset(X, y, val_indices),
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Restore fixed Epoch 2 checkpoint
model = BrainTumorCNN().to(device)

checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold3_epoch2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

assert checkpoint["fold"] == 3
assert checkpoint["epoch"] == 2

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Inference
all_predictions = []
all_actual = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):
        outputs = model(images.to(device))
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            (predictions.cpu().numpy() + 1).tolist()
        )
        all_actual.extend(
            (labels.numpy() + 1).tolist()
        )

        if batch_idx % 50 == 0:
            print(
                f"Inference batch {batch_idx}/{len(val_loader)}"
            )

# Metrics
accuracy = accuracy_score(all_actual, all_predictions)
balanced_acc = balanced_accuracy_score(
    all_actual, all_predictions
)
macro_f1 = f1_score(
    all_actual, all_predictions, average="macro"
)

cm = confusion_matrix(
    all_actual,
    all_predictions,
    labels=[1, 2, 3]
)

print("\nFOLD 3 — FINAL PREDICTION AUDIT")
print("=" * 50)
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        all_actual,
        all_predictions,
        labels=[1, 2, 3],
        target_names=[
            "Meningioma",
            "Glioma",
            "Pituitary"
        ],
        digits=4,
        zero_division=0
    )
)

# Save prediction audit with original metadata
audit = metadata.iloc[val_indices].copy().reset_index(drop=True)

audit["Actual_Label"] = all_actual
audit["Predicted_Label"] = all_predictions
audit["Correct"] = (
    audit["Actual_Label"] == audit["Predicted_Label"]
)

audit_path = os.path.join(
    RESULTS_DIR, "fold3_epoch2_predictions.csv"
)

audit.to_csv(audit_path, index=False)

print("\nPrediction audit saved:")
print(audit_path)

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import time
import gc
import os
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

gc.collect()

model.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

# Validation monitoring
model.eval()

val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

# Save Epoch 1 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold4_epoch1.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 4,
    "epoch": 1,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 4 — EPOCH 1 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import time
import gc
import os
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

gc.collect()

model.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

# Validation monitoring
model.eval()

val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_dataset) if "val_dataset" in globals() else val_loss_total / len(val_loader.dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

# Save predetermined Epoch 2 checkpoint
checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold4_epoch2.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 4,
    "epoch": 2,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 4 — EPOCH 2 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------------
# 1. PATHS AND ENVIRONMENT
# --------------------------------------------------

BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"

X_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_images_224.npy"
)
Y_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_labels.npy"
)
METADATA_PATH = os.path.join(
    BASE_PATH, "results", "verified_dataset_metadata.csv"
)
CHECKPOINT_PATH = os.path.join(
    BASE_PATH, "models", "brain_tumor_cnn_fold4_epoch2.pth"
)
AUDIT_PATH = os.path.join(
    BASE_PATH, "results", "fold4_epoch2_predictions.csv"
)

device = torch.device("cpu")
torch.set_num_threads(4)

# --------------------------------------------------
# 2. LOAD DATA
# --------------------------------------------------

X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata = pd.read_csv(METADATA_PATH)

val_indices = np.where(metadata["Fold"].to_numpy() == 4)[0]

print("FOLD 4 DATA RESTORED")
print("=" * 45)
print("Images:", len(val_indices))
print("Patients:", metadata.iloc[val_indices]["Patient_ID"].nunique())
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2), "GB")

assert len(val_indices) == 628
assert len(set(metadata.iloc[val_indices]["Patient_ID"].astype(str))) == 47

# --------------------------------------------------
# 3. DATASET
# --------------------------------------------------

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label


val_dataset = BrainMRIDataset(X, y, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# --------------------------------------------------
# 4. MODEL ARCHITECTURE
# --------------------------------------------------

class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# --------------------------------------------------
# 5. RESTORE FOLD 4 EPOCH 2 CHECKPOINT
# --------------------------------------------------

model = BrainTumorCNN(num_classes=3).to(device)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

assert checkpoint["fold"] == 4
assert checkpoint["epoch"] == 2

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("\nCHECKPOINT RESTORED")
print("=" * 45)
print("Fold:", checkpoint["fold"])
print("Epoch:", checkpoint["epoch"])
print("Validation accuracy during training:",
      round(checkpoint["validation_accuracy"], 4))

# --------------------------------------------------
# 6. INFERENCE
# --------------------------------------------------

all_predictions = []
all_actual = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(
        val_loader, start=1
    ):

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy() + 1
        )
        all_actual.extend(
            labels.numpy() + 1
        )

        if batch_idx % 50 == 0:
            print(
                f"Inference batch {batch_idx}/{len(val_loader)}"
            )

all_predictions = np.asarray(all_predictions)
all_actual = np.asarray(all_actual)

# --------------------------------------------------
# 7. METRICS
# --------------------------------------------------

accuracy = accuracy_score(all_actual, all_predictions)

balanced_acc = balanced_accuracy_score(
    all_actual, all_predictions
)

macro_f1 = f1_score(
    all_actual, all_predictions, average="macro"
)

cm = confusion_matrix(
    all_actual,
    all_predictions,
    labels=[1, 2, 3]
)

# --------------------------------------------------
# 8. SAVE PREDICTION AUDIT
# --------------------------------------------------

audit = metadata.iloc[val_indices].copy().reset_index(drop=True)

audit["Actual_Label"] = all_actual
audit["Predicted_Label"] = all_predictions
audit["Correct"] = all_actual == all_predictions

audit.to_csv(AUDIT_PATH, index=False)

# --------------------------------------------------
# 9. FINAL REPORT
# --------------------------------------------------

class_names = [
    "Meningioma",
    "Glioma",
    "Pituitary"
]

print("\nFOLD 4 — FINAL PREDICTION AUDIT")
print("=" * 50)
print("Images evaluated:", len(all_predictions))
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(
    classification_report(
        all_actual,
        all_predictions,
        labels=[1, 2, 3],
        target_names=class_names,
        zero_division=0
    )
)

print("\nSaved audit:", AUDIT_PATH)

gc.collect()

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

In [ ]:
import time
import gc
import os
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

gc.collect()

model.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

model.eval()

val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold5_epoch1.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 5,
    "epoch": 1,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 5 — EPOCH 1 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

In [ ]:
import time
import gc
import os
import psutil

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

gc.collect()

model.train()

running_loss = 0.0
correct = 0
total = 0

start_time = time.time()

for batch_idx, (images, labels) in enumerate(train_loader, start=1):

    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad(set_to_none=True)

    outputs = model(images)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    running_loss += loss.item() * images.size(0)

    predictions = outputs.argmax(dim=1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

    if batch_idx % 100 == 0:
        print(f"Training batch {batch_idx}/{len(train_loader)}")

train_loss = running_loss / total
train_accuracy = correct / total

# Validation monitoring
model.eval()

val_loss_total = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss_total += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        if batch_idx % 50 == 0:
            print(f"Validation batch {batch_idx}/{len(val_loader)}")

val_loss = val_loss_total / len(val_dataset)

val_accuracy = accuracy_score(all_labels, all_predictions)
val_balanced_accuracy = balanced_accuracy_score(
    all_labels, all_predictions
)
val_macro_f1 = f1_score(
    all_labels, all_predictions, average="macro"
)

elapsed = time.time() - start_time

checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold5_epoch2.pth"
)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "fold": 5,
    "epoch": 2,
    "learning_rate": optimizer.param_groups[0]["lr"],
    "train_loss": train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy,
    "balanced_accuracy": val_balanced_accuracy,
    "macro_f1": val_macro_f1,
    "class_weights": class_weights.tolist()
}, checkpoint_path)

gc.collect()

print("\nFOLD 5 — EPOCH 2 RESULTS")
print("=" * 45)
print(f"Training loss: {train_loss:.4f}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")
print(f"Balanced accuracy: {val_balanced_accuracy:.4f}")
print(f"Macro F1: {val_macro_f1:.4f}")
print(f"Training time: {elapsed:.2f} seconds")
print("Checkpoint:", checkpoint_path)
print(
    "Available RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import psutil

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

gc.collect()

BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"

X_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_images_224.npy"
)
Y_PATH = os.path.join(
    BASE_PATH, "dataset", "processed", "brain_mri_labels.npy"
)
METADATA_PATH = os.path.join(
    BASE_PATH, "results", "verified_dataset_metadata.csv"
)
MODEL_DIR = os.path.join(BASE_PATH, "models")
RESULTS_DIR = os.path.join(BASE_PATH, "results")

device = torch.device("cpu")
torch.set_num_threads(4)

# Restore data
X = np.load(X_PATH, mmap_mode="r")
y = np.load(Y_PATH)
metadata = pd.read_csv(METADATA_PATH)

fold = 5
val_indices = np.where(metadata["Fold"].values == fold)[0]

# Patient-independent split verification
train_patients = set(
    metadata.loc[metadata["Fold"] != fold, "Patient_ID"].astype(str)
)
val_patients = set(
    metadata.iloc[val_indices]["Patient_ID"].astype(str)
)

assert len(train_patients.intersection(val_patients)) == 0
assert len(val_indices) == 643

class BrainMRIDataset(Dataset):
    def __init__(self, images, labels, indices):
        self.images = images
        self.labels = labels
        self.indices = np.asarray(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        actual_index = self.indices[index]

        image = np.array(
            self.images[actual_index],
            dtype=np.float32,
            copy=True
        )

        image = torch.from_numpy(image).unsqueeze(0)

        label = torch.tensor(
            int(self.labels[actual_index]) - 1,
            dtype=torch.long
        )

        return image, label


val_dataset = BrainMRIDataset(X, y, val_indices)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

# Model architecture
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = BrainTumorCNN(num_classes=3).to(device)

checkpoint_path = os.path.join(
    MODEL_DIR, "brain_tumor_cnn_fold5_epoch2.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

assert checkpoint["fold"] == 5
assert checkpoint["epoch"] == 2

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("FOLD 5 DATA RESTORED")
print("=" * 45)
print("Images:", len(val_indices))
print("Patients:", len(val_patients))
print("Available RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

# Inference
all_predictions = []
all_actual = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(val_loader, start=1):

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy() + 1
        )
        all_actual.extend(
            labels.numpy() + 1
        )

        if batch_idx % 50 == 0:
            print(
                f"Inference batch {batch_idx}/{len(val_loader)}"
            )

# Metrics
accuracy = accuracy_score(all_actual, all_predictions)
balanced_acc = balanced_accuracy_score(
    all_actual, all_predictions
)
macro_f1 = f1_score(
    all_actual, all_predictions, average="macro"
)

cm = confusion_matrix(
    all_actual,
    all_predictions,
    labels=[1, 2, 3]
)

report = classification_report(
    all_actual,
    all_predictions,
    labels=[1, 2, 3],
    target_names=[
        "Meningioma",
        "Glioma",
        "Pituitary"
    ],
    digits=4,
    zero_division=0
)

# Save prediction audit
audit = metadata.iloc[val_indices].copy().reset_index(drop=True)

audit["Actual_Label"] = all_actual
audit["Predicted_Label"] = all_predictions
audit["Correct"] = (
    audit["Actual_Label"] == audit["Predicted_Label"]
)

audit_path = os.path.join(
    RESULTS_DIR, "fold5_epoch2_predictions.csv"
)

audit.to_csv(audit_path, index=False)

print("\nFOLD 5 — FINAL PREDICTION AUDIT")
print("=" * 50)
print("Images evaluated:", len(all_actual))
print(f"Accuracy: {accuracy:.4f}")
print(f"Balanced accuracy: {balanced_acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

print("\nConfusion matrix:")
print(cm)

print("\nClassification report:")
print(report)

print("\nSaved audit:", audit_path)

gc.collect()

print(
    "\nAvailable RAM:",
    round(psutil.virtual_memory().available / (1024**3), 2),
    "GB"
)

In [ ]:
import os
import gc
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
RESULTS_DIR = os.path.join(BASE_PATH, "results")

audit_files = {
    2: "fold2_best_epoch2_predictions.csv",
    3: "fold3_epoch2_predictions.csv",
    4: "fold4_epoch2_predictions.csv",
    5: "fold5_epoch2_predictions.csv"
}

expected_rows = {
    2: 679,
    3: 572,
    4: 628,
    5: 643
}

# Load and validate each fold audit
fold_data = []
fold_results = []

for fold, filename in audit_files.items():

    path = os.path.join(RESULTS_DIR, filename)

    assert os.path.exists(path), f"Missing audit: {path}"

    df = pd.read_csv(path)

    required = [
        "Fold", "Patient_ID", "Actual_Label",
        "Predicted_Label", "Correct"
    ]

    for column in required:
        assert column in df.columns, f"Missing column: {column}"

    assert len(df) == expected_rows[fold], (
        f"Unexpected number of rows in Fold {fold}"
    )

    assert df["Fold"].nunique() == 1
    assert int(df["Fold"].iloc[0]) == fold

    assert set(df["Actual_Label"].unique()).issubset({1, 2, 3})
    assert set(df["Predicted_Label"].unique()).issubset({1, 2, 3})

    # Verify saved correctness values
    calculated_correct = (
        df["Actual_Label"] == df["Predicted_Label"]
    )
    assert np.array_equal(
        calculated_correct.to_numpy(),
        df["Correct"].astype(bool).to_numpy()
    )

    fold_data.append(df)

    actual = df["Actual_Label"]
    predicted = df["Predicted_Label"]

    fold_results.append({
        "Fold": fold,
        "Images": len(df),
        "Patients": df["Patient_ID"].nunique(),
        "Accuracy": accuracy_score(actual, predicted),
        "Balanced_Accuracy": balanced_accuracy_score(
            actual, predicted
        ),
        "Macro_F1": f1_score(
            actual, predicted, average="macro"
        )
    })

# Combine held-out predictions
pooled = pd.concat(fold_data, ignore_index=True)

# Verify total records and patient count
assert len(pooled) == 2522
assert pooled["Patient_ID"].nunique() == 188

# Check duplicate MRI records
if {"Part", "Filename"}.issubset(pooled.columns):
    duplicate_count = pooled.duplicated(
        subset=["Part", "Filename"]
    ).sum()
else:
    duplicate_count = pooled.duplicated(
        subset=["Patient_ID", "Fold", "Actual_Label"]
    ).sum()
    print(
        "Warning: Part/Filename unavailable; "
        "duplicate check uses a less specific key."
    )

assert duplicate_count == 0, "Duplicate records detected!"

# Pooled metrics
actual = pooled["Actual_Label"]
predicted = pooled["Predicted_Label"]

pooled_accuracy = accuracy_score(actual, predicted)
pooled_balanced_accuracy = balanced_accuracy_score(
    actual, predicted
)
pooled_macro_f1 = f1_score(
    actual, predicted, average="macro"
)

pooled_cm = confusion_matrix(
    actual, predicted, labels=[1, 2, 3]
)

# Save per-fold metrics
fold_metrics = pd.DataFrame(fold_results)

fold_metrics_path = os.path.join(
    RESULTS_DIR, "heldout_per_fold_metrics.csv"
)
fold_metrics.to_csv(fold_metrics_path, index=False)

# Save pooled metrics
pooled_metrics = pd.DataFrame([{
    "Evaluation": "Pooled Folds 2-5",
    "Images": len(pooled),
    "Patients": pooled["Patient_ID"].nunique(),
    "Accuracy": pooled_accuracy,
    "Balanced_Accuracy": pooled_balanced_accuracy,
    "Macro_F1": pooled_macro_f1
}])

pooled_metrics_path = os.path.join(
    RESULTS_DIR, "heldout_pooled_metrics.csv"
)
pooled_metrics.to_csv(pooled_metrics_path, index=False)

# Save pooled confusion matrix
cm_df = pd.DataFrame(
    pooled_cm,
    index=["Actual_Meningioma", "Actual_Glioma", "Actual_Pituitary"],
    columns=["Pred_Meningioma", "Pred_Glioma", "Pred_Pituitary"]
)

cm_path = os.path.join(
    RESULTS_DIR, "heldout_pooled_confusion_matrix.csv"
)
cm_df.to_csv(cm_path)

print("HELD-OUT DATA INTEGRITY")
print("=" * 55)
print("Total images:", len(pooled))
print("Unique patients:", pooled["Patient_ID"].nunique())
print("Duplicate records:", duplicate_count)
print("Folds included:", sorted(pooled["Fold"].unique()))

print("\nPER-FOLD PERFORMANCE")
print("=" * 55)
print(fold_metrics.round(4).to_string(index=False))

print("\nPOOLED HELD-OUT PERFORMANCE")
print("=" * 55)
print(f"Accuracy: {pooled_accuracy:.4f}")
print(f"Balanced accuracy: {pooled_balanced_accuracy:.4f}")
print(f"Macro F1: {pooled_macro_f1:.4f}")

print("\nPOOLED CONFUSION MATRIX")
print("=" * 55)
print(pooled_cm)

print("\nPOOLED CLASSIFICATION REPORT")
print("=" * 55)
print(classification_report(
    actual,
    predicted,
    labels=[1, 2, 3],
    target_names=["Meningioma", "Glioma", "Pituitary"],
    digits=4,
    zero_division=0
))

print("\nSAVED RESULTS")
print(fold_metrics_path)
print(pooled_metrics_path)
print(cm_path)

gc.collect()

In [ ]:

import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Project paths
BASE_PATH = r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI"
RESULTS_DIR = os.path.join(BASE_PATH, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Load verified results
fold_metrics = pd.read_csv(
    os.path.join(RESULTS_DIR, "heldout_per_fold_metrics.csv")
)

pooled_metrics = pd.read_csv(
    os.path.join(RESULTS_DIR, "heldout_pooled_metrics.csv")
)

confusion_df = pd.read_csv(
    os.path.join(RESULTS_DIR, "heldout_pooled_confusion_matrix.csv"),
    index_col=0
)

# Load held-out predictions for class-wise metrics
audit_files = [
    "fold2_best_epoch2_predictions.csv",
    "fold3_epoch2_predictions.csv",
    "fold4_epoch2_predictions.csv",
    "fold5_epoch2_predictions.csv"
]

pooled_predictions = pd.concat(
    [
        pd.read_csv(os.path.join(RESULTS_DIR, filename))
        for filename in audit_files
    ],
    ignore_index=True
)

actual = pooled_predictions["Actual_Label"]
predicted = pooled_predictions["Predicted_Label"]

class_names = ["Meningioma", "Glioma", "Pituitary"]

# Verify the data used for figures
assert len(pooled_predictions) == 2522
assert confusion_df.shape == (3, 3)
assert len(fold_metrics) == 4

report = classification_report(
    actual,
    predicted,
    labels=[1, 2, 3],
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

# --------------------------------------------------
# FIGURE 1: POOLED CONFUSION MATRIX
# --------------------------------------------------

cm = confusion_df.to_numpy()

fig, ax = plt.subplots(figsize=(8, 6))

image = ax.imshow(cm, interpolation="nearest")

ax.set_title(
    "Pooled Confusion Matrix — Held-out Evaluation",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Predicted Class", fontsize=11)
ax.set_ylabel("Actual Class", fontsize=11)

ax.set_xticks(range(3))
ax.set_yticks(range(3))

ax.set_xticklabels(class_names, rotation=20, ha="right")
ax.set_yticklabels(class_names)

threshold = cm.max() / 2

for i in range(3):
    for j in range(3):
        ax.text(
            j, i, str(cm[i, j]),
            ha="center",
            va="center",
            color="white" if cm[i, j] > threshold else "black",
            fontsize=12
        )

fig.colorbar(image, ax=ax, label="Number of MRI slices")
fig.tight_layout()

cm_png = os.path.join(
    RESULTS_DIR, "figure_01_pooled_confusion_matrix.png"
)

fig.savefig(cm_png, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# --------------------------------------------------
# FIGURE 2: PERFORMANCE ACROSS FOLDS
# --------------------------------------------------

metrics_to_plot = {
    "Accuracy": "Accuracy",
    "Balanced Accuracy": "Balanced_Accuracy",
    "Macro F1": "Macro_F1"
}

fig, ax = plt.subplots(figsize=(9, 6))

x = np.arange(len(fold_metrics))
width = 0.24

for i, (label, column) in enumerate(metrics_to_plot.items()):
    ax.bar(
        x + (i - 1) * width,
        fold_metrics[column] * 100,
        width,
        label=label
    )

ax.set_title(
    "Model Performance Across Held-out Folds",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Held-out Fold")
ax.set_ylabel("Score (%)")

ax.set_xticks(x)
ax.set_xticklabels(
    [f"Fold {f}" for f in fold_metrics["Fold"]]
)

ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

fold_png = os.path.join(
    RESULTS_DIR, "figure_02_per_fold_performance.png"
)

fig.savefig(fold_png, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# --------------------------------------------------
# FIGURE 3: CLASS-WISE PERFORMANCE
# --------------------------------------------------

precision = [
    report[name]["precision"] * 100
    for name in class_names
]

recall = [
    report[name]["recall"] * 100
    for name in class_names
]

f1 = [
    report[name]["f1-score"] * 100
    for name in class_names
]

fig, ax = plt.subplots(figsize=(9, 6))

x = np.arange(len(class_names))
width = 0.24

ax.bar(x - width, precision, width, label="Precision")
ax.bar(x, recall, width, label="Recall")
ax.bar(x + width, f1, width, label="F1-score")

ax.set_title(
    "Class-wise Performance — Pooled Held-out Evaluation",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Tumor Class")
ax.set_ylabel("Score (%)")

ax.set_xticks(x)
ax.set_xticklabels(class_names)

ax.set_ylim(0, 100)
ax.legend()
ax.grid(axis="y", alpha=0.25)

fig.tight_layout()

class_png = os.path.join(
    RESULTS_DIR, "figure_03_classwise_performance.png"
)

fig.savefig(class_png, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# --------------------------------------------------
# SUMMARY
# --------------------------------------------------

print("\nRESEARCH FIGURES GENERATED")
print("=" * 55)

for path in [cm_png, fold_png, class_png]:
    print(path)
    print(
        "File size:",
        round(os.path.getsize(path) / 1024, 2),
        "KB"
    )

print("\nPooled evaluation figures use:")
print("MRI slices:", len(pooled_predictions))
print("Patients:", pooled_predictions["Patient_ID"].nunique())
print("Held-out folds:", sorted(pooled_predictions["Fold"].unique()))

gc.collect()


In [ ]:
from IPython.display import display, Image
import os

RESULTS_DIR = (
    r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\results"
)

figures = [
    "figure_01_pooled_confusion_matrix.png",
    "figure_02_per_fold_performance.png",
    "figure_03_classwise_performance.png"
]

for filename in figures:
    path = os.path.join(RESULTS_DIR, filename)

    print("\n", filename)
    display(Image(filename=path))

In [ ]:
%pip install streamlit

In [3]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI")
MODEL_PATH = PROJECT_DIR / "models" / "brain_tumor_cnn_weighted_fold1_epoch2.pth"

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_PATH}")

app_code = r'''
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

st.set_page_config(
    page_title="Brain Tumor MRI Classifier",
    page_icon="🧠",
    layout="centered"
)

CLASS_NAMES = ["Meningioma", "Glioma", "Pituitary Tumor"]
CHECKPOINT_PATH = "models/brain_tumor_cnn_weighted_fold1_epoch2.pth"

class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

@st.cache_resource
def load_model():
    model = BrainTumorCNN()
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

st.title("🧠 Brain Tumor MRI Classifier")
st.write(
    "Upload a brain MRI image to obtain an experimental prediction "
    "among three tumor categories."
)

st.info(
    "Supported formats: PNG, JPG, JPEG, BMP and TIFF. "
    "Upload a single 2D MRI slice."
)

uploaded_file = st.file_uploader(
    "Choose an MRI image",
    type=["png", "jpg", "jpeg", "bmp", "tif", "tiff"]
)

if uploaded_file is not None:
    try:
        original_image = Image.open(uploaded_file).convert("L")

        st.subheader("Uploaded MRI")
        st.image(
            original_image,
            caption="Uploaded MRI slice",
            use_container_width=True
        )

        if st.button("Analyze MRI", type="primary", use_container_width=True):
            with st.spinner("Analyzing MRI image..."):
                resized = original_image.resize(
                    (224, 224),
                    resample=Image.Resampling.BILINEAR
                )

                image_array = np.asarray(resized, dtype=np.float32)
                minimum = image_array.min()
                maximum = image_array.max()

                normalized = (
                    (image_array - minimum) /
                    (maximum - minimum + 1e-8)
                )

                input_tensor = torch.from_numpy(normalized).unsqueeze(0).unsqueeze(0)

                model = load_model()

                with torch.no_grad():
                    logits = model(input_tensor)
                    probabilities = torch.softmax(logits, dim=1)[0]

                predicted_index = int(torch.argmax(probabilities))
                predicted_class = CLASS_NAMES[predicted_index]

            st.success("Analysis completed")

            st.subheader("Prediction")
            st.markdown(f"## {predicted_class}")

            st.subheader("Model confidence scores")
            for index, class_name in enumerate(CLASS_NAMES):
                score = float(probabilities[index])
                st.write(f"**{class_name}: {score * 100:.2f}%**")
                st.progress(score)

            st.caption(
                "These scores are model outputs, not calibrated diagnostic "
                "probabilities or medical certainty."
            )

    except Exception as error:
        st.error(f"Could not process this image: {error}")

st.divider()

st.warning(
    "**Research demonstration only.** This experimental model was trained "
    "to choose among Meningioma, Glioma and Pituitary Tumor. It has no "
    "normal/no-tumor class and cannot identify other conditions. "
    "It may produce a tumor prediction even for a normal or unsuitable image. "
    "Do not use its output for diagnosis, treatment or medical decisions."
)

st.caption(
    "Model: experimental Fold 1 development checkpoint | "
    "Input: 224 × 224 grayscale image | Device: CPU"
)
'''

requirements = """streamlit==1.67.0
torch>=2.5,<3
numpy>=1.25,<3
Pillow>=9.2
"""

(PROJECT_DIR / "app.py").write_text(app_code, encoding="utf-8")
(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

print("WEBSITE FILES CREATED")
print("=" * 45)
print("App:", PROJECT_DIR / "app.py")
print("Requirements:", PROJECT_DIR / "requirements.txt")
print("Model checkpoint:", MODEL_PATH)
print("Checkpoint exists:", MODEL_PATH.exists())

WEBSITE FILES CREATED
App: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\app.py
Requirements: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\requirements.txt
Model checkpoint: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\models\brain_tumor_cnn_weighted_fold1_epoch2.pth
Checkpoint exists: True


In [5]:
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

PROJECT_DIR = Path(r"C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI")

X = np.load(
    PROJECT_DIR / "dataset" / "processed" / "brain_mri_images_224.npy",
    mmap_mode="r"
)
y = np.load(PROJECT_DIR / "dataset" / "processed" / "brain_mri_labels.npy")
metadata = pd.read_csv(PROJECT_DIR / "results" / "verified_dataset_metadata.csv")

output_dir = PROJECT_DIR / "results" / "app_test_samples"
output_dir.mkdir(parents=True, exist_ok=True)

class_names = {
    1: "Meningioma",
    2: "Glioma",
    3: "Pituitary"
}

for label, name in class_names.items():
    indices = np.where(
        (metadata["Fold"].to_numpy() == 1) & (y == label)
    )[0]

    index = int(indices[0])
    image = (np.clip(X[index], 0, 1) * 255).astype(np.uint8)

    path = output_dir / f"test_{name.lower()}.png"
    Image.fromarray(image).save(path)

    print(f"{name}: {path}")

print("\nThree test images created.")

Meningioma: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\results\app_test_samples\test_meningioma.png
Glioma: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\results\app_test_samples\test_glioma.png
Pituitary: C:\Medical_Imaging_Research\Project_01_Brain_Tumor_MRI\results\app_test_samples\test_pituitary.png

Three test images created.
